# Serving headroom and critical path diagnostic
See EXPERIMENT-CARD.md; no gameplay efficacy claim.


In [ ]:
"""First notebook cell: retain actual inventory and reject the wrong accelerator."""
import json
import subprocess
from pathlib import Path

hardware_dir = Path('/kaggle/working/headroom')
hardware_dir.mkdir(parents=True, exist_ok=True)
inventory = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader,nounits'],
    capture_output=True, text=True, timeout=15)
rows = [line.strip().split(',') for line in inventory.stdout.splitlines() if line.strip()]
valid = (inventory.returncode == 0 and len(rows) == 1 and len(rows[0]) == 3
         and 'RTX PRO 6000' in rows[0][0]
         and rows[0][1].strip().isdigit() and int(rows[0][1]) >= 90000)
receipt = {'run_id': 'serving-headroom-v2-20261008', 'returncode': inventory.returncode,
           'stdout': inventory.stdout, 'stderr': inventory.stderr, 'valid': valid}
(hardware_dir / 'hardware.json').write_text(json.dumps(receipt, indent=2) + '\n')
print('retry hardware:', json.dumps(receipt), flush=True)
if not valid:
    raise RuntimeError('Retry requires exactly one RTX PRO 6000 with >=90000 MiB; abort before loading')


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

NOTEBOOK_START_TIME = time.time()
SESSION_DEADLINE = time.monotonic() + 3300  # self-imposed cap (CARD.md operations)
WHEELHOUSE_DIR    = '/kaggle/input/datasets/dfranzen/pennyroyal-v253'
MODEL_DIR         = '/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1'
DRAFT_MODEL_DIR   = '/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1'
SERVED_MODEL_NAME = 'flashnext'
SERVED_MODEL_HOST = '127.0.0.1'
SERVED_MODEL_PORT = 8001
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
OUT = WORKING_DIR / 'headroom'
OUT.mkdir(exist_ok=True)

# replay dataset: locate by its manifest, re-hash every file, refuse to run on a mismatch
cands = [p.parent for p in Path('/kaggle/input').rglob('manifest.json') if (p.parent / 'replay.py').is_file()]
assert len(cands) == 1, f'expected one replay dataset, found {cands}'
DATA = cands[0]
assert hashlib.sha256((DATA / 'manifest.json').read_bytes()).hexdigest() == 'c21852dc33715d0efe6b4d62b70d7869040b859b4f1c2ec36ad1ea22e64b9713', 'dataset differs from the frozen notebook'
prov = json.loads((DATA / 'provenance.json').read_text())
assert prov['code_commit'] == '28ef795f16f5165dc6c71426569bf7a16968d970', 'dataset code commit differs from the frozen notebook'
assert not prov['code_dirty'], 'production payload must come from committed code'
manifest = json.loads((DATA / 'manifest.json').read_text())
def _sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()
readback = {k: _sha(DATA / k) for k in manifest}
bad = sorted(k for k in manifest if readback[k] != manifest[k])
(OUT / 'readback.json').write_text(json.dumps({'data_dir': str(DATA), 'files': readback, 'mismatches': bad}, indent=1))
assert not bad, f'dataset readback mismatch: {bad}'
print(f'replay dataset {DATA}: {len(readback)} files, hashes match manifest')
NSYS = shutil.which('nsys')
print('nsys:', NSYS or 'absent -> profile deferred')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'])


In [ ]:
import os
import threading
import time

from concurrent.futures import as_completed, ThreadPoolExecutor

def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except: pass

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

precache_wheels_thread = threading.Thread(target=precache, args=(WHEELHOUSE_DIR,))
precache_wheels_thread.start()
precache_wheels_thread.join()

precache_model_thread = threading.Thread(target=precache, args=(MODEL_DIR, DRAFT_MODEL_DIR), kwargs={'delay': 60, 'threads': 3})

In [ ]:
# Paste this entire file into the launcher cell.
# Required notebook variables: MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR,
# WORKING_DIR, SERVED_MODEL_NAME, SERVED_MODEL_PORT.
# MODEL_DIR: Intel AutoRound INT4 target (BF16 PLE or albucino FP8 PLE).
# DRAFT_MODEL_DIR: downloaded albucino runtime/mtp-int4-g32, or its dataset root.
import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error
import copy, hashlib, json, re, shlex, socket, zipfile
from pathlib import Path

PREFIX = "/tmp/sgl-intel"       # Separate from the previous fork's /tmp/sgl venv.
LOG = str(Path(WORKING_DIR) / "serve.log")
CFG = dict(
    CTX=(116+12+8)*1024,
    MEMFRAC=0.96,
    MAXREQ=10,
    CUDAGRAPH_MAXBS=10,
    MAMBA_CACHE=60,
    CHUNK=8192,
    MAX_PREFILL=16384,
    SPEC=True,
    FRSPEC=True,
    AUTOTUNE=True,
    # Preserve the original Kaggle checkpoint prefetch/cache behavior.
    PREFETCH_CHECKPOINTS=True,
    DROP_CACHE_AFTER_LOAD=True,
    LINEAR_BACKEND="flashinfer",
    KVDTYPE="fp8_e4m3",
    SSM_DTYPE="bfloat16",
    MAMBA_RADIX="extra_buffer",
    GDN_MTP_CACHE_MODE="none",
    SERVED_NAME=SERVED_MODEL_NAME,
    SPEC_ACCEPT_SINGLE=1.0,
    SPEC_ACCEPT_ACC=1.0,
    SPEC_STEPS=3,
)
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
SGLANG = f"{VENV}/bin/sglang"
TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


def prepare_draft_view(target, source_root, work_root):
    """Adapt metadata only; link original dataset weights without copying them."""
    target, source_root, work_root = map(Path, (target, source_root, work_root))
    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
    candidates = []
    for path in configs:
        cfg = json.loads(path.read_text())
        q = cfg.get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            candidates.append((path.parent, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
    source, cfg = candidates[0]
    index, shards = indexed_shards(source)
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("Draft index does not contain the expected MTP experts")
    cfg = copy.deepcopy(cfg)
    q = cfg["quantization_config"]
    group = q["config_groups"]["mtp_routed_experts"]
    w = group["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError("Expected symmetric INT4 group32 draft experts")
    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
        raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
        raise RuntimeError("Unexpected draft ignore rules")
    group["targets"] = [EXPERT_TARGET]
    q["ignore"] = [DENSE_IGNORE]
    # Content/path-specific directory makes notebook reruns idempotent without
    # overwriting an unrelated view. We never modify the read-only datasets.
    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
    view = work_root / f"draft-view-{suffix}"
    view.mkdir(parents=True, exist_ok=True)
    config_path = view / "config.json"
    config_text = json.dumps(cfg, indent=2) + "\n"
    if config_path.exists() or config_path.is_symlink():
        if config_path.is_symlink() or config_path.read_text() != config_text:
            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
    else:
        config_path.write_text(config_text)
    links = {name: source / name for name in shards}
    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
        if (target / name).is_file(): links[name] = target / name
    for name, src in links.items():
        dest = view / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            if not dest.is_symlink() or dest.resolve() != src.resolve():
                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
        else:
            dest.symlink_to(src.resolve())
    print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
    return view


# ---- validate notebook inputs / identify the offline bundle ----
MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0
# Refuse to start a second server on an occupied port; do not kill other jobs.
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
        raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
Path(PREFIX).mkdir(parents=True, exist_ok=True)
Path(LOG).parent.mkdir(parents=True, exist_ok=True)
WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
wheel = sglang_wheels[0]
install_marker = Path(PREFIX) / "installed-bundle.txt"

# ---- one-time offline install, isolated from the old mratsim environment ----
install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
install_env.pop("PYTHONPATH", None)
if not Path(SGLANG).exists() or not install_marker.is_file():
    print("=== installing Pennyroyal (offline) ===")
    # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
    uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
    assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
    uv_binary = Path(PREFIX) / "bin/uv"
    uv_binary.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uv_wheels[0]) as archive:
        members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
        assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
        binary = archive.read(members[0])
        assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
        uv_binary.write_bytes(binary)
    uv_binary.chmod(0o755)
    uv = [str(uv_binary)]
    if not Path(PYTHON).exists():
        run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
    install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
    run(install + ["-r", str(LOCK)], env=install_env)
    run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
    install_marker.write_text('install')
else:
    print(f"Matching offline bundle already installed at {VENV}")

# ---- bundled CUDA toolkit and linker symlinks ----
cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
    raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
CUDA_HOME = str(cu13[0])
libdir = Path(CUDA_HOME) / "lib"
lib64 = Path(CUDA_HOME) / "lib64"
if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
for so in sorted(libdir.glob("*.so.*")):
    link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
    if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
if not (libdir / "libcuda.so").exists():
    candidates = []
    for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
        candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
    if not candidates:
        code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
        maps = run([PYTHON, "-c", code], env=install_env).stdout
        candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
    if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
    link = libdir / "libcuda.so"
    if link.is_symlink(): link.unlink()
    link.symlink_to(candidates[0].resolve())
CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
assert CXX, "Host C++ compiler not found"
CC = CXX.replace("g++", "gcc")
assert shutil.which(CC), f"Matching C compiler missing: {CC}"

# ---- runtime environment ----
env = dict(os.environ)
env.pop("PYTHONPATH", None)
cache = f"{PREFIX}/cache"
for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
    Path(cache, name).mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
    "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
    "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
    "TORCH_CUDA_ARCH_LIST": "12.0",
    "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
    "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
    "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
    "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
    "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
    "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
    "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
    "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
    "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
    "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
    "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
    "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
    "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
})
run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
print("Host compiler:", CXX)
# Report the optional later caching patch. This cell does not modify wheel code.
run([PYTHON, "-c", '''import pathlib, sglang, torch
print("sglang", sglang.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "CUDA unavailable"
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
'''], env=env)

# ---- validate target and create the read-only draft's SGLang view ----
target_cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
q = target_cfg.get("quantization_config", {})
assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "MODEL_DIR must be the Intel/albucino AutoRound INT4 checkpoint"
_, target_shards = indexed_shards(MODEL_DIR)
print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype:", target_cfg.get("text_config", {}).get("ple_embedding_dtype", "checkpoint default (typically BF16)"))
DRAFT_VIEW = None
if CFG["SPEC"]:
    assert Path(DRAFT_MODEL_DIR).is_dir(), f"draft dir not found: {DRAFT_MODEL_DIR}"
    DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)

# ---- assemble Pennyroyal / AutoRound launch arguments ----
graph_bs = sorted({1, 2, 4, 7, 8, 9, 10, CFG["MAXREQ"], CFG["CUDAGRAPH_MAXBS"]})
graph_bs = [n for n in graph_bs if n <= CFG["CUDAGRAPH_MAXBS"]]
args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
    "--model-loader-extra-config", '{"enable_multithread_load":false}',
    "--served-model-name", str(CFG["SERVED_NAME"]), "--host", "127.0.0.1",
    "--port", str(SERVED_MODEL_PORT), "--tensor-parallel-size", "1",
    "--dtype", "bfloat16", "--quantization", "auto-round",
    "--kv-cache-dtype", CFG["KVDTYPE"], "--mem-fraction-static", str(CFG["MEMFRAC"]),
    "--context-length", str(CFG["CTX"]), "--page-size", "64",
    "--max-running-requests", str(CFG["MAXREQ"]), "--chunked-prefill-size", str(CFG["CHUNK"]),
    "--max-prefill-tokens", str(CFG["MAX_PREFILL"]),
    "--cuda-graph-max-bs-decode", str(CFG["CUDAGRAPH_MAXBS"]),
    "--cuda-graph-bs-decode", *map(str, graph_bs),
    "--mamba-ssm-dtype", CFG["SSM_DTYPE"], "--max-mamba-cache-size", str(CFG["MAMBA_CACHE"]),
    "--mamba-radix-cache-strategy", CFG["MAMBA_RADIX"], "--mamba-track-interval", "64",
    "--mamba-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-decode-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-prefill-backend", CFG["LINEAR_BACKEND"],
    "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
    "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
    "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
    "--default-chat-template-kwargs", '{"preserve_thinking":true}',
    "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
    "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging"]
if CFG["PREFETCH_CHECKPOINTS"]: args += ["--weight-loader-prefetch-checkpoints"]
if CFG["DROP_CACHE_AFTER_LOAD"]: args += ["--weight-loader-drop-cache-after-load"]
tmpl = Path(MODEL_DIR, "chat_template.jinja")
if tmpl.is_file(): args += ["--chat-template", str(tmpl)]
if not CFG["AUTOTUNE"]: args += ["--disable-flashinfer-autotune"]
if CFG["GDN_MTP_CACHE_MODE"]: args += ["--gdn-mtp-cache-mode", CFG["GDN_MTP_CACHE_MODE"]]
if CFG["SPEC"]:
    args += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
        "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"]+1),
        "--speculative-draft-model-path", str(DRAFT_VIEW),
        "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
        "--speculative-draft-kv-cache-dtype", CFG["KVDTYPE"],
        "--speculative-accept-threshold-single", str(CFG["SPEC_ACCEPT_SINGLE"]),
        "--speculative-accept-threshold-acc", str(CFG["SPEC_ACCEPT_ACC"])]
    if CFG["FRSPEC"]:
        tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
        if tok is None: tok = find_unique(WHEELHOUSE_DIR, "flash-next-64k.pt")
        assert sha256(tok) == TOKEN_MAP_SHA, "Wrong FR-Spec map; use the map from the pinned Pennyroyal bundle"
        assert sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA, "Tokenizer differs from the FR-Spec map; set CFG['FRSPEC']=False"
        args += ["--speculative-token-map", str(tok)]

print('incumbent server args assembled; launches are driven by the sweep')


In [ ]:
run([PYTHON, '-c', "import hashlib, json, shutil\nfrom pathlib import Path\nimport sglang, torch\nroot = Path(sglang.__file__).parent\nout = Path('/kaggle/working/headroom/runtime-source'); out.mkdir(parents=True, exist_ok=True)\nselected = ['srt/managers/scheduler_components/profiler_manager.py',\n 'srt/managers/scheduler_components/metrics_reporter.py', 'srt/managers/io_struct.py',\n 'srt/models/qwen4_exp.py', 'srt/models/qwen4_exp_mtp.py',\n 'srt/speculative/eagle_worker_v2.py', 'srt/layers/quantization/gptq/schemes/gptq_marlin.py',\n 'srt/observability/metrics_collector.py', 'srt/mem_cache/kv_cache_configurator.py',\n 'srt/managers/scheduler_components/batch_result_processor.py', 'srt/managers/scheduler.py',\n 'srt/entrypoints/http_server.py']\nreceipt = {'sglang':sglang.__version__, 'torch':torch.__version__, 'files':{}}\nfor relative in selected:\n    source = root / relative\n    if source.is_file():\n        dest = out / relative; dest.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(source, dest)\n        receipt['files'][relative] = hashlib.sha256(source.read_bytes()).hexdigest()\n    else: receipt['files'][relative] = 'absent'\n(out / 'identity.json').write_text(json.dumps(receipt, indent=1))\nprint(json.dumps(receipt))\n"], env=env)
run([PYTHON, '-c', "import hashlib, pathlib, sglang\nroot = pathlib.Path(sglang.__file__).parent\nexpected = {'srt/observability/metrics_collector.py': '6400f15a91a52b96d8a206b8794d1aaf4ab71c4c5d9ae1c7027aa651d6ff9591', 'srt/mem_cache/kv_cache_configurator.py': 'a759ae971d6da987cf11bf81de85d6f67d577fc487bb52bf241c4019a825a138', 'srt/managers/scheduler_components/metrics_reporter.py': '6149a4ae4ebc2318517f8583915255de4f3549064d4fd5a4caaf015a48aafc02', 'srt/managers/scheduler_components/profiler_manager.py': '8e4a29923065c2c674151e94d1d4355ee295f1fca0104d36d5de70e5f7e22fe2', 'srt/managers/scheduler_components/batch_result_processor.py': '8c9626605546de2bc54d5f25a7ae16caeb9ec96e273af50adf91076df543e05b', 'srt/managers/scheduler.py': 'a1209592d8146328c11f64e338d3448438428260ab9c45e1702347663d917ffd', 'srt/entrypoints/http_server.py': '833d5da3572305a96df3607833256ee31f6e7a03d7ea91c260a8bde076846ca6'}\nfor name, digest in expected.items():\n    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == digest, 'native contract differs: ' + name\nprint('Native measurement/admission/profile source hashes match qualification')\n"], env=env)
shutil.copy2(Path(MODEL_DIR) / 'config.json', OUT / 'target-config.json')


In [ ]:
sys.path.insert(0, str(DATA))
"""One-load diagnostic. Embedded verbatim in the notebook; stdlib only."""
import concurrent.futures
import hashlib
import gzip
from http.client import HTTPConnection
import json
import math
import re
import statistics
import time
from pathlib import Path

import payload
import replay
import sim

RUN_ID = "serving-headroom-v2-20261008"
PLAN = [("PROFILE1", 1), ("PROFILE10", 10), ("N10A", 10), ("N10B", 10)]
PORT = 8001


def http(path, body=None, timeout=30):
    conn = HTTPConnection("127.0.0.1", PORT, timeout=timeout)
    try:
        conn.request("GET" if body is None else "POST", path,
                     body=None if body is None else json.dumps(body).encode(),
                     headers={"Content-Type": "application/json"})
        response = conn.getresponse()
        raw = response.read()
        if response.status != 200:
            raise RuntimeError(f"{path}: HTTP {response.status}: {raw[:500]!r}")
        return raw
    finally:
        conn.close()


def sample(directory=None):
    raw = http("/metrics")
    vals = parse_metrics(raw)
    if directory is not None:
        path = directory / "metrics" / f"{time.monotonic_ns()}.prom"
        path.parent.mkdir(exist_ok=True)
        path.write_bytes(raw)
        vals.update(raw_metrics=str(path.relative_to(directory)), raw_sha256=hashlib.sha256(raw).hexdigest())
    return vals


def parse_metrics(raw):
    vals = {}
    wanted = {**replay.WANTED, "sglang:num_retracted_requests_total": "retractions",
              "sglang:token_usage": "kv_fraction", "sglang:mamba_usage": "mamba_fraction"}
    for line in raw.decode().splitlines():
        match = replay.METRIC.match(line)
        if match and match.group(1) in wanted:
            key = wanted[match.group(1)]
            vals[key] = vals.get(key, 0.) + float(match.group(3))
        if match and match.group(1) == "sglang:realtime_tokens_total":
            labels = match.group(2) or ""
            for mode, key in (("decode", "decode_tokens"), ("prefill_compute", "prefill_compute"),
                              ("prefill_cache", "prefill_cache")):
                if re.search(r'(?:\{|,)mode="' + mode + r'"(?:,|\})', labels):
                    if key in vals:
                        raise RuntimeError("multiple scheduler series for realtime mode: " + mode)
                    vals[key] = float(match.group(3))
    if any(key in vals for key in ("decode_tokens", "prefill_compute", "prefill_cache")):
        # The pinned collector only instantiates a mode after its first nonzero increment.
        vals.setdefault("prefill_compute", 0.)
        vals.setdefault("prefill_cache", 0.)
    return vals


def steady(a, b, n):
    """Require actual occupancy and no realtime prefill, including cache hits."""
    required = ("running", "queue", "pending", "prefill_compute", "prefill_cache")
    if any(key not in x or not math.isfinite(x[key]) for x in (a, b) for key in required):
        return False
    return (a["t"] >= 5 and b["t"] > a["t"] and
            all(x["running"] == n and x["queue"] == 0 and x["pending"] == n for x in (a, b)) and
            all(a[key] == b[key] for key in ("prefill_compute", "prefill_cache")))


def profile_ready(samples, n):
    # Independent of token throughput: a broken counter must not suppress diagnosis.
    return (len(samples) >= 4 and samples[-1]["t"] - samples[-4]["t"] >= 3 and
            all(steady(a, b, n) for a, b in zip(samples[-4:], samples[-3:])))


def plateau(samples, n):
    """Exclude prefill, changing occupancy, the first 5s, and counter resets."""
    intervals = []
    counters = [x["decode_tokens"] for x in samples if "decode_tokens" in x]
    first_seen = next((i for i, x in enumerate(samples) if "decode_tokens" in x), len(samples))
    if (any("decode_tokens" not in x for x in samples[first_seen:]) or
        any(not math.isfinite(x) for x in counters) or any(b < a for a, b in zip(counters, counters[1:]))):
        return {"valid": False, "seconds": 0, "U": None, "CV": None,
                "intervals": [], "all_qualifying_intervals": [], "reason": "missing, nonfinite or reset decode counter"}
    for a, b in zip(samples, samples[1:]):
        if not steady(a, b, n):
            continue
        dt = b["t"] - a["t"]
        delta = b.get("decode_tokens", math.nan) - a.get("decode_tokens", math.nan)
        if dt > 0 and math.isfinite(delta) and delta >= 0:
            intervals.append({"start": a["t"], "end": b["t"], "dt": dt,
                              "tokens": delta, "rate": delta / dt})
    # Preserve disjoint segments and select the longest contiguous segment.
    segments = []
    for row in intervals:
        if not segments or segments[-1][-1]["end"] != row["start"]:
            segments.append([])
        segments[-1].append(row)
    segment = max(segments, key=lambda xs: sum(x["dt"] for x in xs), default=[])
    seconds = sum(x["dt"] for x in segment)
    # Counters update in bursts. Compare >=5s windows rather than scrape ticks.
    rates, pending = [], []
    for row in segment:
        pending.append(row)
        elapsed = sum(x["dt"] for x in pending)
        if elapsed >= 5:
            rates.append(sum(x["tokens"] for x in pending) / elapsed)
            pending = []
    mean = statistics.mean(rates) if rates else None
    return {"valid": len(segment) >= 5 and seconds >= 8 and mean is not None and mean > 0, "seconds": seconds,
            "U": sum(x["tokens"] for x in segment) / seconds if seconds else None,
            "CV": statistics.pstdev(rates) / mean if mean and len(rates) >= 2 else None,
            "intervals": segment, "all_qualifying_intervals": intervals}


def cohort(data):
    games = sim.build(json.loads((data / "workload.json").read_text()))
    chosen = sorted((g["reqs"][0]["prompt_tokens"], name, g["reqs"][0]["line_index"])
                    for name, g in games.items())[:10]
    bodies = replay.Bodies(data, [x[1] for x in chosen])
    result = []
    for prompt, game, index in chosen:
        body = payload.build(bodies.get(game, index), "flashnext", 4096)
        body.update(ignore_eos=True, seed=20261008)
        result.append({"game": game, "index": index, "recorded_prompt": prompt, "body": body})
    return result


def generate(job, directory, ev, max_tokens):
    game = job["game"]
    body = dict(job["body"], max_tokens=max_tokens)
    path = directory / (game + "-request.json")
    path.write_text(json.dumps(body))
    ev.emit("send", game=game, request_sha256=hashlib.sha256(path.read_bytes()).hexdigest())
    t0 = time.monotonic()
    try:
        raw = http("/v1/chat/completions", body, timeout=600)
        (directory / (game + "-response.json")).write_bytes(raw)
        value = json.loads(raw)
        usage = value["usage"]
        length = usage["completion_tokens"]
        cached = (usage.get("prompt_tokens_details") or {}).get("cached_tokens", 0)
        rec = {"game": game, "usage": usage, "elapsed": time.monotonic() - t0,
               "cache_fraction": cached / usage["prompt_tokens"], "length_ok": length == max_tokens}
        ev.emit("response", **rec)
        if length != max_tokens:
            raise RuntimeError(f"{game}: expected {max_tokens}, got {length}")
        return rec
    except Exception as error:
        ev.emit("request_error", game=game, error=repr(error))
        raise


def profile_block(label, out, ev):
    req = {"output_dir": str(out / "traces"), "num_steps": 64, "activities": ["CPU", "GPU"],
           "with_stack": False, "record_shapes": False, "profile_id": label,
           "profile_by_stage": False, "detailed_annotations": True}
    ev.emit("profile_request", request=req)
    raw = http("/start_profile", req, timeout=120)
    ev.emit("profile_response", raw=raw.decode())
    if raw.strip() != b"Start profiling.":
        raise RuntimeError(f"{label}: unexpected profiler response: {raw!r}")


def verify_trace(label, out):
    paths = sorted((out / "traces").glob(label + "-*.trace.json.gz"))
    if not paths:
        raise RuntimeError(f"{label}: profiler produced no trace")
    receipts = []
    for path in paths:
        with gzip.open(path, "rt") as stream:
            events = json.load(stream)["traceEvents"]
        kernels = [e for e in events if e.get("cat") == "kernel" and e.get("ph") == "X" and e.get("dur", 0) > 0]
        if not kernels:
            raise RuntimeError(f"{label}: trace contains no CUDA kernels")
        receipts.append({"path": str(path.relative_to(out)), "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                         "kernel_events": len(kernels)})
    return receipts


def capacity(info, jobs):
    if len(jobs) != 10:
        raise RuntimeError("expected ten fixed workload prefixes")
    states = info.get("internal_states", [])
    if len(states) != 1 or states[0].get("effective_max_running_requests_per_dp") != 10:
        raise RuntimeError("expected one scheduler with effective max-running=10")
    state = states[0]
    # Conservative sum: do not rely on unknown inter-request prefix sharing.
    required = sum(((j["recorded_prompt"] + 4096 + 4 + 63) // 64) * 64 for j in jobs)
    available = state.get("memory_usage", {}).get("token_capacity", 0)
    if available < required:
        raise RuntimeError(f"insufficient KV capacity: {available} < {required}")
    return {"effective_max_running": 10, "token_capacity": available, "required_tokens": required,
            "max_mamba_cache_size": state.get("max_mamba_cache_size")}


def accounting(before, samples, after, total, n):
    counts = [before, *samples, after]
    for key in ("decode_tokens", "gen_tokens", "retractions"):
        first = next((i for i, x in enumerate(counts) if key in x), len(counts))
        values = [x[key] for x in counts[first:] if key in x]
        if (any(key not in x for x in counts[first:]) or
            any(not math.isfinite(x) or x < 0 for x in values) or
            any(b < a for a, b in zip(values, values[1:]))):
            raise RuntimeError("invalid counter lifetime: " + key)
    completion = after.get("gen_tokens", 0) - before.get("gen_tokens", 0)
    decode = after.get("decode_tokens", 0) - before.get("decode_tokens", 0)
    retractions = after.get("retractions", 0) - before.get("retractions", 0)
    # Overlap can count delayed work on finished requests, as well as first-token
    # and final-chunk differences. This is a predeclared error tolerance, not an
    # exact physical bound or a correction fitted after the run. Warmup is diagnostic.
    tolerance = .02*total if total >= 4096*n else None
    if completion != total or (tolerance is not None and abs(decode - total) > tolerance) or retractions != 0:
        raise RuntimeError(f"wire/counter reconciliation failed: completion={completion}, decode={decode}, wire={total}, retractions={retractions}")
    return {"completion_delta": completion, "decode_delta": decode,
            "wire_tokens": total, "decode_boundary_tolerance": tolerance,
            "decode_relative_difference": decode/total-1, "retractions": retractions}


def batch(label, jobs, out, max_tokens=4096):
    directory = out / label
    directory.mkdir()
    n = len(jobs)
    ev = replay.Events(directory / "events.jsonl")
    ev.emit("header", label=label, n=n, max_tokens=max_tokens)
    samples, profile_attempted = [], False
    result = {"label": label, "n": n, "profile": label.startswith("PROFILE")}
    start = time.monotonic()
    try:
        before = sample(directory)
        ev.emit("metrics_before", **before)
        with concurrent.futures.ThreadPoolExecutor(max_workers=n) as pool:
            futures = [pool.submit(generate, job, directory, ev, max_tokens) for job in jobs]
            while not all(f.done() for f in futures):
                if time.monotonic() - start > 600:
                    raise TimeoutError("batch deadline")
                vals = ev.emit("metrics", pending=sum(not f.done() for f in futures), **sample(directory))
                samples.append(vals)
                if result["profile"] and not profile_attempted and profile_ready(samples, n):
                    profile_attempted = True
                    profile_block(label, out, ev)
                time.sleep(1)
            responses = [f.result() for f in futures]
        result.update(status="complete", responses=responses, plateau=plateau(samples, n),
                      cache_ok=all(x["cache_fraction"] >= .9 for x in responses),
                      wall_seconds=time.monotonic() - start, profile_attempted=profile_attempted)
        after = sample(directory)
        ev.emit("metrics_after", **after)
        result["accounting"] = accounting(before, samples, after, sum(x["usage"]["completion_tokens"] for x in responses), n)
        result["retractions"] = result["accounting"]["retractions"]
        if result["profile"]:
            result["traces"] = verify_trace(label, out)
        if label != "WARMUP" and (not result["plateau"]["valid"] or not result["cache_ok"] or result["retractions"] != 0):
            raise RuntimeError(f"{label}: decode/profile qualification failed")
        ev.emit("finished", **result)
        return result
    except Exception as error:
        result.update(status="failed", error=repr(error), plateau=plateau(samples, n))
        ev.emit("failed", **result)
        raise
    finally:
        (directory / "summary.json").write_text(json.dumps(result, indent=1))
        ev.close()


def run_probe(data, out, deadline):
    jobs = cohort(data)
    (out / "cohort.json").write_text(json.dumps([{k: v for k, v in j.items() if k != "body"} for j in jobs], indent=1))
    raw_info = http("/server_info")
    (out / "server-info-before.json").write_bytes(raw_info)
    state = {"run_id": RUN_ID, "blocks": []}
    try:
        state["capacity"] = capacity(json.loads(raw_info), jobs)
        # Fixed prefix warmup is excluded from throughput. No tokenization refit.
        state["warmup"] = batch("WARMUP", jobs, out, max_tokens=32)
        for label, n in PLAN:
            if deadline - time.monotonic() < 660:
                state["stop"] = "insufficient deadline for bounded next batch"
                break
            print("headroom block:", label, flush=True)
            state["blocks"].append(batch(label, jobs[:n], out))
            (out / "state.json").write_text(json.dumps(state, indent=1))
        (out / "server-info-after.json").write_bytes(http("/server_info"))
    except Exception as error:
        state["stop"] = repr(error)
        raise
    finally:
        (out / "state.json").write_text(json.dumps(state, indent=1))

import signal
import sweep
sys.path.insert(0, str(DATA))
proc = None
def cap_handler(signum, frame):
    raise TimeoutError('headroom notebook deadline')
signal.signal(signal.SIGALRM, cap_handler)
signal.alarm(max(1, int(SESSION_DEADLINE-time.monotonic())))
try:
    precache_model_thread.start()
    (OUT / 'server-args.json').write_text(json.dumps(args))
    (OUT / 'server-env.json').write_text(json.dumps({k:v for k,v in env.items() if k.startswith(('SGLANG_', 'CUDA_', 'TORCH_', 'OMP_', 'MKL_'))}))
    with (OUT / 'server.log').open('ab', buffering=0) as server_log:
        proc = subprocess.Popen(args, env=env, stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
        ready, why = sweep.wait_health(proc, 'http://127.0.0.1:8001', 1800)
        (OUT / 'startup.json').write_text(json.dumps({'ready':ready,'why':why,'pid':proc.pid}))
        if not ready: raise RuntimeError(why)
        run_probe(DATA, OUT, SESSION_DEADLINE)
finally:
    signal.alarm(0)
    if proc is not None: sweep.stop_process_group(proc, grace=30)
    manifest = {str(p.relative_to(OUT)): _sha(p) for p in OUT.rglob('*') if p.is_file()}
    (OUT / 'output-hashes.json').write_text(json.dumps(manifest, indent=1))
